In [ ]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 加载数据
carseats = sm.datasets.get_rdataset("Carseats", "ISLR").data

# 对连续自变量进行中心化
carseats['Price_c'] = carseats['Price'] - carseats['Price'].mean()
carseats['Income_c'] = carseats['Income'] - carseats['Income'].mean()
carseats['Advertising_c'] = carseats['Advertising'] - carseats['Advertising'].mean()

# 建立多元线性回归模型
model = smf.ols('Sales ~ Price_c + Income_c + Advertising_c + ShelveLoc', data=carseats).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  Sales   R-squared:                       0.627
Model:                            OLS   Adj. R-squared:                  0.622
Method:                 Least Squares   F-statistic:                     132.4
Date:                Sun, 20 Sep 2026   Prob (F-statistic):           4.89e-82
Time:                        16:38:37   Log-Likelihood:                -785.22
No. Observations:                 400   AIC:                             1582.
Df Residuals:                     394   BIC:                             1606.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept               5.4307    

- **基准组**：`ShelveLoc[Bad]`
- **ShelveLoc[Good] 商业含义**：在控制价格、收入和广告投入不变的情况下，货架位置为 Good 的商店，其销售额平均比 Bad 货架（基准组）高出该系数值

In [8]:
# 生成哑变量并加入常数项
X = pd.get_dummies(carseats[['Price_c', 'Income_c', 'Advertising_c', 'ShelveLoc']], 
                   columns=['ShelveLoc'], drop_first=True).astype(float)
X['Intercept'] = 1.0

# 计算 VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif_data)

           Variable       VIF
0           Price_c  1.007733
1          Income_c  1.012422
2     Advertising_c  1.009404
3    ShelveLoc_Good  1.497279
4  ShelveLoc_Medium  1.493782
5         Intercept  4.191157


- 判断标准：VIF > 10 为严重共线性，VIF > 5 为中等风险。
- 结论：所有变量 VIF < 5（或严格 < 10），则模型不存在严重的多重共线性风险，系数估计稳定。